In [42]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [43]:
import pandas as pd
import json
from pathlib import Path


In [44]:
ROLE_SKILL_PROFILES_PATH = Path("skill_gap/role_skill_profiles.csv")

df = pd.read_csv(ROLE_SKILL_PROFILES_PATH)
df.head()


,role_id,skill_id,frequency,importance
0,AI_ML_ENGINEER_INT,SK001,16,0.020627
1,AI_ML_ENGINEER_INT,SK002,3,0.003868
2,AI_ML_ENGINEER_INT,SK003,21,0.027073
3,AI_ML_ENGINEER_INT,SK004,73,0.146752
4,AI_ML_ENGINEER_INT,SK006,2,0.007306


In [45]:
df["role_id"].unique()


array(['AI_ML_ENGINEER_INT', 'DATA_ANALYST_INT', 'DATA_ENGINEER_INT',
       'DEVOPS_TRAINEE', 'INTERN_SE', 'JR_BE_DEV', 'JR_BUSINESS_ANALYST',
       'JR_FE_DEV', 'JR_FS_DEV', 'JR_IT_SUPPORT', 'JR_MOBILE_DEV',
       'JR_QA_ENG', 'JR_SE', 'JR_SYS_ADMIN', 'JR_UI_UX_DESIGNER'],
      dtype=object)

In [46]:
ROLE_LEVEL_ORDER = {
    "INTERN": 0,
    "JR": 1,
    "JUNIOR": 1,
    "INT": 2,
    "MID": 2,
    "": 2,          # default mid
    "SR": 3,
    "SENIOR": 3,
    "LEAD": 4,
    "PRINCIPAL": 5,
    "ARCHITECT": 6
}


In [47]:
def parse_role(role_id: str):
    parts = role_id.split("_")

    level = ""
    if parts[-1] in ROLE_LEVEL_ORDER:
        level = parts[-1]
        family = "_".join(parts[:-1])
    else:
        family = role_id

    return family, level


In [48]:
df[["role_family", "role_level"]] = df["role_id"].apply(
    lambda r: pd.Series(parse_role(r))
)

df.head()


,role_id,skill_id,frequency,importance,role_family,role_level
0,AI_ML_ENGINEER_INT,SK001,16,0.020627,AI_ML_ENGINEER,INT
1,AI_ML_ENGINEER_INT,SK002,3,0.003868,AI_ML_ENGINEER,INT
2,AI_ML_ENGINEER_INT,SK003,21,0.027073,AI_ML_ENGINEER,INT
3,AI_ML_ENGINEER_INT,SK004,73,0.146752,AI_ML_ENGINEER,INT
4,AI_ML_ENGINEER_INT,SK006,2,0.007306,AI_ML_ENGINEER,INT


In [49]:
df["level_rank"] = df["role_level"].map(
    lambda x: ROLE_LEVEL_ORDER.get(x, 2)
)

df[["role_id", "role_family", "role_level", "level_rank"]].sort_values(
    ["role_family", "level_rank"]
).head(20)


,role_id,role_family,role_level,level_rank
0,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
1,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
2,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
3,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
4,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
5,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
6,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
7,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
8,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2
9,AI_ML_ENGINEER_INT,AI_ML_ENGINEER,INT,2


In [50]:
career_ladders = {}

for family, group in df.groupby("role_family"):
    ordered = (
        group.sort_values("level_rank")
        .drop_duplicates("role_id")
    )

    ladder = ordered["role_id"].tolist()

    if len(ladder) >= 2:
        career_ladders[family] = ladder


In [51]:
list(career_ladders.items())[:5]


[]

In [52]:
OUTPUT_PATH = Path("career_path/career_ladders_derived.json")

with open(OUTPUT_PATH, "w") as f:
    json.dump(career_ladders, f, indent=2)

print("Saved:", OUTPUT_PATH)


Saved: career_path/career_ladders_derived.json


In [53]:
with open("career_path/career_ladders_derived.json") as f:
    CAREER_LADDERS = json.load(f)
